In [ ]:
import os
import datetime
import colormaps
from pathlib import Path
from functools import partial
from itertools import product
from string import ascii_lowercase
from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.colors import (
    LinearSegmentedColormap,
    BoundaryNorm,
    Normalize,
    rgb_to_hsv,
    hsv_to_rgb,
)
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs

os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import (
    PRETTIER_VARNAME,
    YEARS,
    UNITS,
    RESULTS,
    FACTORS_UNITS,
    FACTORS,
    DATADIR,
    DUNCANS_REGIONS_NAMES,
    MONTH_NAMES,
    FIGURES,
    RADIUS,
    get_region,
    squarify,
    polars_to_xarray,
    xarray_to_polars,
)
from jetutils.data import standardize, standardize_polars_dtypes, compute_anomalies_pl
from jetutils.geospatial import (
    central_diff,
    haversine_from_dl,
    compute_relative_clim,
    compute_relative_sm,
    compute_relative_std,
    compute_relative_anom,
    create_all_relative_plots,
)
from jetutils.jet_finding import (
    average_jet_categories,
    track_jets,
    slowness_from_cross,
    connected_from_cross,
    slowness_expr,
    spells_from_cross_catd_simple,
    spells_from_cross,
    extract_features,
    find_all_jets,
    jet_position_as_da,
    to_one_large,
)
from jetutils.plots import (
    STYLE_SHEET,
    COLORS,
    COLORS_EXT,
    WERNLI_FLAIR,
    WERNLI_FLAIR_LEVELS,
    Clusterplot,
    plot_interp,
    plot_relative_time,
    gaussian_kde,
    plot_seasonal,
)
from jetutils.anyspell import (
    make_daily,
    mask_from_spells_pl,
    subset_around_onset,
    extend_spells,
    get_spells,
)
from jetutils.stats import create_bootstrapped_times, odds_ratio, is_signif_OR, common_OR

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

year_bounds = {
    "historical": (1970, 2009),
    "ssp370": (2060, 2099),
}

summers = {}
summer_filters = {}
JETS = ["STJ", "EDJ"]
RUNS = list(year_bounds)
JET_COLORS = [COLORS[2], COLORS[1]]
for key, bounds in year_bounds.items():
    ALL_TIMES = (
        pl.datetime_range(
            start=pl.datetime(bounds[0], 1, 1),
            end=pl.datetime(bounds[1] + 1, 1, 1),
            closed="left",
            interval="6h",
            eager=True,
            time_unit="ms",
        )
        .rename("time")
        .to_frame()
    )
    summer_filter = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9]))
    summer = summer_filter["time"]
    summers[key] = summer
    summer_filters[key] = summer_filter
    
    

In [ ]:

# for run in ["historical", "ssp370"]:
#     path = Path(DATADIR, "CESM2/high_wind", run, "results/1")
#     jets = pl.read_parquet(path.joinpath("jets.parquet"))
#     jets_newcat = is_polar_gmix(jets, ("s", "theta"), mode="week", n_init=20, init_params="random_from_data", v2=True, use_prev=True)
#     jets_newcat = jets_newcat.rename({"is_polar": "is_polar_old", "is_polar_right": "is_polar"})
    
#     newpath = Path(DATADIR, "CESM2/high_wind", run, "results/2")
#     newpath.mkdir(exist_ok=True)
#     jets = jets.drop("diff", "is_polar_old")
#     jets.write_parquet(newpath.joinpath("jets.parquet"))
#     props = pl.read_parquet(path.joinpath("props.parquet"))
#     newcat = jets.group_by("member", "time", "jet ID").agg(pl.col("is_polar").mean())
#     props = props.drop("is_polar").join(newcat, on=("member", "time", "jet ID"))
#     props.write_parquet(newpath.joinpath("props.parquet"))
#     phat_jets = to_one_large(jets, int_EDJ_threshold=1.3e8)
#     cross = track_jets(phat_jets)
#     cross.write_parquet(newpath.joinpath("cross.parquet"))
#     pers = pers_from_cross(cross)    
#     phat_filter = (pl.col("is_polar") < 0.5) | ((pl.col("is_polar") > 0.5) & (pl.col("int") > 1.3e8))
#     phat_props = props.filter(phat_filter)
#     phat_props = average_jet_categories(phat_props, polar_cutoff=0.5)

In [ ]:
both_props = {}
both_cross = {}
both_summary_comps = {}
both_summer_summary_comps = {}
both_spells = {}
dist_threshold = 4.0e6
overlap_threshold = 0.6
dis_polar_thresh = 0.12
force_summary_comp = False
for run in RUNS:
    basepath = Path(DATADIR, run)
    jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
    cross_opath = basepath.joinpath("cross.parquet")
    if cross_opath.is_file():
        cross = pl.read_parquet(cross_opath)
    else:
        cross = track_jets(jets)
        cross.write_parquet(cross_opath)   
    both_cross[run] = cross
    props = pl.read_parquet(basepath.joinpath("props.parquet"))
    summary_comp_file = basepath.joinpath("summary_comp.parquet")
    if summary_comp_file.is_file() and not force_summary_comp:
        summary_comp = pl.read_parquet(summary_comp_file)
    else:
        _, summary_comp = connected_from_cross(jets, cross, dist_threshold, overlap_threshold, dis_polar_thresh)
        jet_column = (
            pl.when((pl.col("is_polar") > 0.5).mean().over("spell") > 0.5)
            .then(pl.lit("EDJ"))
            .otherwise(pl.lit("STJ"))
        )
        summary_comp = (
            summary_comp
            .drop("s", "theta", "is_polar", "s_right", "theta_right", "is_polar_right")
            .join(props["member", "time", "jet ID", "is_polar"], on=("member", "time", "jet ID"))
            .with_columns(
                jet=jet_column,
                slowness=slowness_expr(),
            )
            .with_columns(
                slowness_sum=pl.col("slowness").sum().over("member", "spell"),
            )
            .sort("member", "time", "jet ID")
        )
        summary_comp.write_parquet(summary_comp_file)
    props = props.join(summary_comp["member", "time", "jet ID", "slowness", "slowness_sum"], on=["member", "time", "jet ID"], how="left")
    both_props[run] = props
    spells_list = spells_from_cross(
        jets,
        cross,
        summary_comp,
        dist_threshold,
        overlap_threshold,
        dis_polar_thresh,
        n_STJ=30,
        n_EDJ=30,
        season=summers[run],
        STJ_lat_threshold=30,
    )
    
    both_spells[run] = spells_list
    for jet_name, spells in spells_list.items():
        print(run, jet_name, spells["spell"].n_unique())
        print(run, jet_name, spells["len"].min())
        both_spells[f"{run}_{jet_name}"] = spells
    summary_comp = summary_comp.filter(pl.col("len") > 1)
    summer_summary_comp = summary_comp.filter(
        pl.col("time")
        .is_in(pl.lit(summers[run].implode().first(), pl.List(pl.Datetime("ms"))))
        .over("member", "spell")
    )
    both_summary_comps[run] = summary_comp
    both_summer_summary_comps[run] = summer_summary_comp
    # both_phat_props[run] = pl.read_parquet(basepath.joinpath("props_full.parquet"))
    # cross_catd = pl.read_parquet(basepath.joinpath("cross.parquet"))
    # break

In [ ]:
figure = plt.figure(figsize=(12, 9), constrained_layout=True)
subfigs = figure.subfigures(2, 2)
subfigs = subfigs.ravel()

def annotate(ax, annotation, coords=None):
    if coords is None:
        coords = (0.97, 0.97)
    ax.annotate(
        annotation,
        coords,
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontweight="demi",
        bbox={
            "boxstyle": "square, pad=0.1",
            "edgecolor": "none",
            "facecolor": "white",
        },
        usetex=False,
    )
    return ax


# distances, slowness_sum, lifetime lengths, episode distrib
jet_name_expr = (
    pl.when(pl.col("is_polar") < 0.5).then(pl.lit("STJ")).otherwise(pl.lit("EDJ"))
)
colors = [COLORS[2], COLORS[1]]
letters_list = np.split(np.array(list(ascii_lowercase))[:24], 6)

## cross distances, no smoothing
fig = subfigs[0]
axes = fig.subplots(2, 2, sharex="all", sharey="all")
axes = axes.T
letters = letters_list[0]
for run, axs, letters_ in zip(RUNS, axes, np.split(letters, 2)):
    df = both_summer_summary_comps[run]
    cross_summer = summer_filters[run].join(both_cross[run], on="time")
    cross_summer = cross_summer.filter(pl.col("dis_polar") < dis_polar_thresh).with_columns(
        jet=jet_name_expr
    )
    for jet_name, color, ax, letter in zip(JETS, colors, axs, letters_):
        df_ = cross_summer.filter(pl.col("jet") == jet_name)
        ax.hist(
            df_["dist"] / 1e6,
            color=color,
            bins=np.linspace(0, 10, 31),
            density=True,
            edgecolor="white",
            linewidth=1,
        )
        ax.axvline(dist_threshold / 1e6, color="black", ls="dashed")
        ax = annotate(ax, letter)
fig.supxlabel("Dist. between jets [1000 km]")
fig.supylabel("Distance density")
axes[1, 0].set_xticks(np.arange(11))

## Persistence
fig = subfigs[1]
axes = fig.subplots(2, 2, sharex="all", sharey="all")
axes = axes.T
letters = letters_list[1]
for run, axs, letters_ in zip(RUNS, axes, np.split(letters, 2)):
    df = both_summer_summary_comps[run]
    for jet_name, color, ax, letter in zip(JETS, colors, axs, letters_):
        df_ = (
            df.group_by("spell")
            .agg(pl.col("slowness_sum").first(), pl.col("jet").first())
            .filter(pl.col("jet") == jet_name)
        )
        ax.hist(
            df_["slowness_sum"],
            color=color,
            bins=np.linspace(0, 3, 31),
            density=True,
            log=True,
            edgecolor="white",
            linewidth=1,
        )

        cutoff = both_spells[f"{run}_{jet_name}"]["slowness_sum"].min()
        ax.axvline(cutoff, color="black", ls="dashed")
        ax = annotate(ax, letter)
fig.supxlabel("Persistence [s/m]")
fig.supylabel("Persistence density")
axes[1, 0].set_xticks(np.linspace(0, 3, 7))

## Lengths
fig = subfigs[2]
axes = fig.subplots(2, 2, sharex="all", sharey="all")
axes = axes.T
letters = letters_list[2]
for run, axs, letters_ in zip(RUNS, axes, np.split(letters, 2)):
    df = both_summer_summary_comps[run]
    for jet_name, color, ax, letter in zip(JETS, colors, axs, letters_):
        df_ = (
            df.group_by("spell")
            .agg(pl.col("len").first(), pl.col("jet").first())
            .filter(pl.col("jet") == jet_name)
        )
        ax.hist(
            df_["len"],
            color=color,
            bins=np.arange(2, 15) - 0.5,
            density=False,
            log=True,
            edgecolor="white",
            linewidth=1,
        )

        cutoff = both_spells[f"{run}_{jet_name}"]["len"].min()
        ax.axvline(cutoff, color="black", ls="dashed")
        ax = annotate(ax, letter)
fig.supxlabel("Lifetime [day]")
fig.supylabel("Lifetime bin count")
axes[1, 0].set_xticks(np.arange(1, 15))

## When PEs
fig = subfigs[3]
axes = fig.subplots(2, 2, sharex="all", sharey="all")
axes = axes.T
letters = letters_list[3]
bins = pl.datetime_range(
    datetime.datetime(1959, 6, 1),
    datetime.datetime(1959, 10, 1),
    "7d",
    closed="left",
    eager=True,
)
for run, axs, letters_ in zip(RUNS, axes, np.split(letters, 2)):
    huh = (
        summers[run].dt.ordinal_day()
        .unique()
        .sort()
        .to_frame()
        .with_columns(
            time_2=pl.datetime(year=1959, month=1, day=1) + pl.duration(days="time")
        )
    )
    for jet_name, color, ax, letter in zip(JETS, colors, axs, letters_):
        df_ = both_spells[f"{run}_{jet_name}"].select(
            time=pl.datetime(year=1959, month=1, day=1, time_unit="ms")
            + (
                pl.col("time")
                - pl.datetime(year=pl.col("time").dt.year(), month=1, day=1, time_unit="ms")
            )
        )
        ax.hist(df_["time"], bins=bins, color=color, edgecolor="white", linewidth=1)
        ax = annotate(ax, letter)

    ax.xaxis.set_major_formatter(DateFormatter("%m-%d"))
    ax.xaxis.set_tick_params(labelsize=11, rotation=30)
    ticks = ax.get_xticks()
    ticklabels = ax.get_xticklabels()
    ax.set_xticks(ticks, labels=[t.get_text() for t in ticklabels], ha="right")
fig.supylabel("Episode bin count")

# figure.savefig(f"{FIGURES}/ClimateChange/stats.pdf")

In [ ]:
_ = plot_seasonal(average_jet_categories(both_props["historical"]), ["mean_lat", "mean_s", "tilt", "wavinessR16", "slowness", "slowness_sum"], nrows=2, ncols=3, clear=False, save=False)

In [ ]:
_ = plot_seasonal(average_jet_categories(both_props["ssp370"]), ["mean_lat", "mean_s", "tilt", "wavinessR16", "slowness", "slowness_sum"], nrows=2, ncols=3, clear=False, save=False)

In [ ]:
from matplotlib.dates import MonthLocator
from jetutils.data import periodic_rolling_pl
from matplotlib.lines import Line2D
data_vars: list = ["mean_lat", "mean_s", "mean_theta", "wavinessR16", "slowness", "slowness_sum"]
nrows: int = 2
ncols: int = 3

plot_kwargs = {"historical": [average_jet_categories(both_props["historical"]), "solid"], "ssp370": [average_jet_categories(both_props["ssp370"]), "dashed"]}

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols * 3.5, nrows * 2.4),
    constrained_layout=True,
    sharex="all",
)
axes = axes.flatten()
jets = average_jet_categories(both_props["historical"])["jet"].unique().to_numpy()
njets = len(jets)
gb_args = [pl.col("time").dt.ordinal_day().alias("dayofyear"), pl.col("jet")]
aggs = [
    pl.col(col).replace([float("-inf"), float("inf"), float("nan")], None).mean()
    for col in data_vars
]


for name, args in plot_kwargs.items():
    props, ls = args
    gb = props.group_by(gb_args)

    means = gb.agg(aggs).sort("dayofyear", "jet", descending=[False, True])
    means = squarify(means, ["dayofyear", "jet"])
    means = periodic_rolling_pl(means, 15, data_vars)

    x = means["dayofyear"].unique()
    color_order = [2, 1]
    for letter, varname, ax in zip(ascii_lowercase, data_vars, axes.ravel()):
        dji = varname == "double_jet_index"
        factor = FACTORS.get(varname, 1)
        if factor == 1:
            factor_str = ""
        else:
            factor_str = str(int(np.log10(np.abs(factor))))
            factor_str = r"$10^{" + factor_str + r"} \times $"
        ys = means[varname].to_numpy().reshape(366, njets)
        for i in range(njets):
            factor = factor / (1 + 9 * (varname in ["slowness", "slowness_sum"] and i == 1))
            color = "black" if dji else COLORS[color_order[i]]
            ax.plot(
                x,
                ys[:, i] / factor,
                lw=3,
                color=color,
                zorder=10,
                ls=ls,
            )
            if dji:
                break
        ax.set_title(
            f"{letter}) {PRETTIER_VARNAME.get(varname, varname)} [{factor_str}{UNITS.get(varname, '')}]"
        )
        ax.xaxis.set_major_locator(MonthLocator(range(0, 13, 3)))
        ax.xaxis.set_major_formatter(DateFormatter("%b"))
        ax.set_xlim(min(x), max(x))
        if varname == "mean_lev":
            ax.invert_yaxis()
        # if name == "dobl":
        #     ylim = ax.get_ylim()
        #     wherex = np.isin(x, JJASDOYS)
        #     ax.fill_between(
        #         x, *ylim, where=wherex, alpha=0.1, color="black", zorder=-10
        #     )
        #     ax.set_ylim(ylim)
           
linedata =  [[0, 1], [0, 1]]
handles = [
    Line2D(*linedata, lw=2, ls="solid", color=COLORS[2]),
    Line2D(*linedata, lw=2, ls="solid", color=COLORS[1]),
    Line2D(*linedata, lw=2, ls="solid", color="black"),
    Line2D(*linedata, lw=2, ls="dashed", color="black"),
]
labels = [*JETS, *RUNS]
axes.ravel()[1].legend(ncol=2, handles=handles, labels=labels, labelspacing=0.2, handletextpad=0.2, columnspacing=0.01, handlelength=1.1).set_zorder(102)
# fig.savefig(f"{FIGURES}/Diabatic/seasonal.pdf")

In [ ]:
diffs = (
    average_jet_categories(both_props["historical"])
    .group_by("member", "jet", dayofyear=pl.col("time").dt.ordinal_day())
    .agg(cs.numeric().mean().cast(pl.Float32()))
    .join(
        average_jet_categories(both_props["ssp370"])
        .group_by("member", "jet", dayofyear=pl.col("time").dt.ordinal_day())
        .agg(cs.numeric().mean().cast(pl.Float32())),
        on=["dayofyear", "jet"],
        suffix="_future"
    )
    .select("jet", "dayofyear", *[(pl.col(f"{col}_future") - pl.col(col)).alias(col) for col in both_props["historical"].columns if col not in ["member", "time", "jet ID"]])
)

In [ ]:
means = diffs.group_by("jet", "dayofyear").agg(cs.exclude("member", "member_future").mean()).sort("jet", "dayofyear")
stds = diffs.group_by("jet", "dayofyear").agg(cs.exclude("member", "member_future").std()).sort("jet", "dayofyear")
medians = diffs.group_by("jet", "dayofyear").agg(cs.exclude("member", "member_future").median()).sort("jet", "dayofyear")
q1 = diffs.group_by("jet", "dayofyear").agg(cs.exclude("member", "member_future").quantile(0.2)).sort("jet", "dayofyear")
q2 = diffs.group_by("jet", "dayofyear").agg(cs.exclude("member", "member_future").quantile(0.8)).sort("jet", "dayofyear")
for var in ["mean_lat", "mean_s", "mean_theta", "wavinessR16", "slowness", "slowness_sum"]:
    plt.figure()
    for jet, color in zip(JETS, JET_COLORS):
        means_ = means.filter(pl.col("jet") == jet)
        stds_ = stds.filter(pl.col("jet") == jet)
        q1_ = q1.filter(pl.col("jet") == jet)
        q2_ = q2.filter(pl.col("jet") == jet)
        medians_ = medians.filter(pl.col("jet") == jet)
        plt.fill_between(means_["dayofyear"], q1_[var], q2_[var], alpha=0.2, color=color)
        plt.plot(means_["dayofyear"], means_[var], color=color, lw=4)
        plt.plot(means_["dayofyear"], medians_[var], color=color, ls="dashed")
    plt.suptitle(var)